# depth  —  Sec 3.2  one corrupted layer of 28 + 7-model depth check

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# Depth sweep — is the tolerable-set wall *stronger* than depth compounding?

The paper explains near-zero radius (**O1**) and fidelity$\perp$usability (**O2**) via a
per-layer factor $\pi<1$ compounding as $\pi^{L}$. But $\pi^{L}$ is only the *depth-scaling*
of the wall. It cannot alone explain why the **rank-1** nearest genuine cache already craters
(needs large *per-layer* sharpness at $C^*$), nor why **cos 0.99** transfers nothing (needs
*anisotropy*: the sensitive axis is ~orthogonal to what cosine sees). We decompose the wall.

| leg | question | isolates |
|---|---|---|
| **A** depth-budget sweep | corrupt only first $\ell$ layers, sweep $\ell$ | $\pi^{\ell}$ (the depth leg) |
| **B** single-layer radius | is the radius already small at $\ell=1$? | per-layer sharpness |
| **C** anisotropy probe | fixed cos budget, aligned vs random direction | $O2$, **depth-independent** |
| **D** genuine radius vs $L$ | oracle/nn1 protocol across real 6–36-layer models | the literal "5-layer" rebuttal |

Read: if **B** kills the rank-1 neighbour after *one* layer, and **C** shows equal-cosine
perturbations with opposite transfer, then depth is *necessary not sufficient* — the wall is
per-layer sharpness $\times$ anisotropy, *amplified* by depth.

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)

# Primary reader for legs A/B/C:
BASE="Qwen/Qwen2.5-1.5B"
# Leg D depth ladder (same-family Pythia + Qwen; GPT2 as shallow out-of-family anchor):
LADDER=["EleutherAI/pythia-70m","EleutherAI/pythia-410m","EleutherAI/pythia-1.4b",
        "Qwen/Qwen2.5-0.5B","Qwen/Qwen2.5-1.5B","Qwen/Qwen2.5-3B","gpt2"]
CFG=dict(
  bank_ctx=120, bank_cap=6000,
  n_eval=200, max_ctx=256, gen_len=8,
  cos_budgets=[0.999,0.995,0.99,0.98,0.95,0.90],   # leg C fidelity budget
  aniso_rank=8,                                    # top-k sensitive directions
  ladder_neval=120,
)
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(bank_ctx=30,bank_cap=1500,n_eval=24,ladder_neval=24)
    LADDER=["EleutherAI/pythia-70m","Qwen/Qwen2.5-0.5B","gpt2"]
    print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_depthfast" if os.environ.get("KVL_FAST","0")=="1" else "_depth"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)
RESPATH=f"results/depth{TAG}.json"
RES={"config":CFG,"base":BASE,"ladder":LADDER,"legs":{}}

## 1. Model harness — load ANY causal LM as a frozen reader with GQA-aware KV tooling

In [ ]:
class Reader:
    '''Wrap a frozen HF causal LM with the oracle rig's cache/RoPE/decode tooling,
    generalised to arbitrary L, n_kv, n_q. Handles models with RoPE (Qwen/Pythia)
    and learned-positional (GPT2) alike, because we always decode via past_key_values
    and let the model apply its own positions through position_ids where supported.'''
    def __init__(self, name):
        self.name=name
        self.tok=AutoTokenizer.from_pretrained(name,cache_dir=CACHE)
        if self.tok.pad_token is None: self.tok.pad_token=self.tok.eos_token
        self.m=AutoModelForCausalLM.from_pretrained(name,cache_dir=CACHE,
                 torch_dtype=torch.float32).to(DEVICE).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        ids=self.tok("hi",return_tensors="pt").input_ids.to(DEVICE)
        with torch.no_grad(): pkv=self.m(ids,use_cache=True).past_key_values
        if hasattr(pkv,"to_legacy_cache"): pkv=pkv.to_legacy_cache()
        self.L=len(pkv); _,self.n_kv,_,self.d_head=pkv[0][0].shape
        cfg=self.m.config
        self.n_q=getattr(cfg,"num_attention_heads",self.n_kv)
        self.group=max(1,self.n_q//self.n_kv)
        self.kdim=self.n_kv*self.d_head
        self.rope=hasattr(cfg,"rope_theta") or "gpt_neox" in type(self.m).__name__.lower() \
                  or "qwen" in name.lower() or "pythia" in name.lower()
        print(f"[{name}] L={self.L} n_q={self.n_q} n_kv={self.n_kv} group={self.group} "
              f"d_head={self.d_head} rope={self.rope}",flush=True)
    def native_ids(self,text,max_len):
        return self.tok([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
    def kv_stack(self,past):
        outs=[]
        for (k,v) in past:
            B,H,T,dh=k.shape
            outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                                   v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
        return torch.stack(outs,0)
    @torch.no_grad()
    def ctx_cache(self,text,max_len):
        ids=self.native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
        leg=self.m(body,use_cache=True).past_key_values
        if hasattr(leg,"to_legacy_cache"): leg=leg.to_legacy_cache()
        return self.kv_stack(leg), last, body
    def slots_to_legacy(self,vec):
        L,B,Kk,_=vec.shape; half=self.kdim; leg=[]
        for l in range(L):
            kt=vec[l,:,:,:half].reshape(B,Kk,self.n_kv,self.d_head).permute(0,2,1,3).contiguous()
            vt=vec[l,:,:,half:].reshape(B,Kk,self.n_kv,self.d_head).permute(0,2,1,3).contiguous()
            leg.append((kt,vt))
        return leg
    def to_past(self,vec): return DynamicCache.from_legacy_cache(tuple(self.slots_to_legacy(vec)))
    def _pos(self,s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
    @torch.no_grad()
    def greedy(self,prefix_vec,last_id):
        past=self.to_past(prefix_vec); T0=prefix_vec.shape[2]
        try:
            o=self.m(last_id,past_key_values=past,position_ids=self._pos(T0,1),
                     attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
        except TypeError:
            o=self.m(last_id,past_key_values=past,use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; Lc=T0+1
        for _ in range(CFG["gen_len"]-1):
            try:
                o=self.m(cur,past_key_values=past,position_ids=self._pos(Lc,1),
                         attention_mask=torch.ones(1,Lc+1,device=DEVICE),use_cache=True)
            except TypeError:
                o=self.m(cur,past_key_values=past,use_cache=True)
            past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); Lc+=1
        return self.tok.decode(torch.cat(gen,1)[0])

## 2. Split-context SQuAD (identical protocol to oracle)

In [ ]:
def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
train_qa=build_qa(CFG["bank_ctx"],"train"); eval_qa=build_qa(CFG["n_eval"],"validation")
print("train",len(train_qa),"eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False

## 3. Bank + perturbation primitives (for the reader R)

`nn1(vec)`: replace every token's key/value by its **rank-1 nearest** genuine bank
entry (a real, on-manifold key from a *different* token) — the O1 stimulus.
`gauss_to_cos(vec, target_cos)`: add isotropic noise scaled so mean per-layer key
cosine to the original hits `target_cos` — a fidelity-matched *random* perturbation.
`aniso_to_cos(...)`: perturb along the leading right-singular directions of the
per-layer attention-logit Jacobian $dL/dK$ at $C^*$, to the same cosine budget.

In [ ]:
def build_bank(R):
    Kb=[[] for _ in range(R.L)]; Vb=[[] for _ in range(R.L)]
    for it in train_qa[:CFG["bank_ctx"]]:
        vec,_,_=R.ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
        for l in range(R.L):
            Kb[l].append(vec[l,0,:,:R.kdim]); Vb[l].append(vec[l,0,:,R.kdim:])
    def cap(lst):
        out=[]
        for l in range(R.L):
            X=torch.cat(lst[l],0)
            if X.shape[0]>CFG["bank_cap"]:
                idx=torch.randperm(X.shape[0],device=X.device)[:CFG["bank_cap"]]; X=X[idx]
            out.append(X.contiguous())
        return out
    Kbank,Vbank=cap(Kb),cap(Vb)
    Kn=[Kbank[l]/(Kbank[l].norm(dim=1,keepdim=True)+1e-9) for l in range(R.L)]
    return Kbank,Vbank,Kn

@torch.no_grad()
def nn1(R,vec,Kbank,Vbank,Kn):
    out=vec.clone()
    for l in range(R.L):
        Ks=vec[l,0,:,:R.kdim]; K1=Ks/(Ks.norm(dim=1,keepdim=True)+1e-9)
        j=(K1@Kn[l].T).argmax(1)
        out[l,0,:,:R.kdim]=Kbank[l][j]; out[l,0,:,R.kdim:]=Vbank[l][j]
    return out

@torch.no_grad()
def _match_cos(orig,delta,target_cos):
    # scale delta so cosine(orig, orig+delta) == target_cos (per-tensor, closed form on the flat vector)
    o=orig.reshape(-1); d=delta.reshape(-1)
    d=d-(d@o)/(o@o+1e-12)*o                      # orthogonalise delta wrt orig
    no=o.norm()+1e-12; nd=d.norm()+1e-12
    # cos = 1/sqrt(1+ (|s*d|/|o|)^2)  => |s*d| = |o|*sqrt(1/cos^2 -1)
    tgt=no*float(np.sqrt(max(1e-9,1.0/target_cos**2-1.0)))
    d=d*(tgt/nd)
    return (o+d).reshape(orig.shape)

@torch.no_grad()
def gauss_to_cos(R,vec,target_cos,gen=None):
    out=vec.clone()
    for l in range(R.L):
        K=vec[l,0,:,:R.kdim]
        noise=torch.randn_like(K)
        out[l,0,:,:R.kdim]=_match_cos(K,noise,target_cos)
    return out

### 3b. Readout-sensitivity directions (the anisotropy axis)

For a batch of eval contexts we form the reader's genuine per-layer keys $K$ and queries
$Q$ at $C^*$, compute attention logits $L=QK^\top/\sqrt d$, and take the gradient of the
scalar $\lVert\text{softmax}(L)\rVert^2$ (a proxy for "which key coordinates move the
attention distribution") wrt $K$. The leading singular directions of the stacked gradient
are the **high-leverage** axes. Perturbing along them at fixed cosine is the aligned arm;
random Gaussian at the same cosine is the control.

In [ ]:
def sensitivity_dirs(R,items,rank):
    '''Return, per layer, an orthonormal (rank x kdim) basis of top readout-sensitive
    directions in KEY space, estimated over a few contexts.'''
    accum=[torch.zeros(R.kdim,R.kdim,device=DEVICE) for _ in range(R.L)]
    nseen=0
    layers=R.m.model.layers if hasattr(R.m,"model") and hasattr(R.m.model,"layers") else None
    for it in items[:min(len(items),24)]:
        text=make_prompt(it["full"],it["q"])
        ids=R.native_ids(text,CFG["max_ctx"]); body=ids[:,:-1]
        # capture hidden states -> recompute K,Q per layer via the model's own proj if available
        with torch.no_grad():
            out=R.m(body,use_cache=True,output_hidden_states=True)
        hs=out.hidden_states  # tuple(L+1) of (1,T,hid)
        if layers is None:
            return None
        for l in range(R.L):
            attn=layers[l].self_attn
            h=hs[l].detach()
            try:
                Kt=attn.k_proj(h); Qt=attn.q_proj(h)
            except Exception:
                return None
            T=Kt.shape[1]
            K=Kt.view(1,T,R.n_kv,R.d_head).permute(0,2,1,3)          # (1,nkv,T,dh)
            Q=Qt.view(1,T,R.n_q,R.d_head).permute(0,2,1,3)           # (1,nq,T,dh)
            Kr=K.repeat_interleave(R.group,1)
            K=K.clone().requires_grad_(True)
            Krg=K.repeat_interleave(R.group,1)
            scale=1.0/(R.d_head**0.5)
            Lg=torch.einsum('bhtd,bhsd->bhts',Q,Krg)*scale
            mask=torch.triu(torch.ones(T,T,device=DEVICE),1).bool()
            Lg=Lg.masked_fill(mask,float("-inf"))
            P=torch.softmax(Lg,-1)
            s=(P*P).sum()
            g,=torch.autograd.grad(s,K,retain_graph=False)
            # g: (1,nkv,T,dh) -> per-(kv-head,token) row in kdim space (concat heads)
            G=g.permute(0,2,1,3).reshape(-1,R.kdim)                  # (T, kdim)
            accum[l]+=G.T@G
        nseen+=1
    dirs=[]
    for l in range(R.L):
        M=accum[l]/max(1,nseen)
        evals,evecs=torch.linalg.eigh(M)                            # ascending
        V=evecs[:,-rank:]                                           # top-rank
        dirs.append(V.contiguous())                                # (kdim, rank)
    return dirs

@torch.no_grad()
def aniso_to_cos(R,vec,target_cos,dirs):
    out=vec.clone()
    for l in range(R.L):
        K=vec[l,0,:,:R.kdim]                                        # (T,kdim)
        V=dirs[l]                                                   # (kdim,rank)
        coeff=torch.randn(K.shape[0],V.shape[1],device=DEVICE)
        delta=coeff@V.T                                             # in sensitive subspace
        out[l,0,:,:R.kdim]=_match_cos(K,delta,target_cos)
    return out

## 4. Depth-budget injection: corrupt first l layers, keep genuine tail

In [ ]:
@torch.no_grad()
def depth_budget_vec(R,own_vec,pert_vec,l):
    '''First l layers from pert_vec (corrupted), layers l..L-1 from own_vec (genuine).'''
    out=own_vec.clone()
    if l>0: out[:l]=pert_vec[:l]
    return out

## 5. Transfer eval (floor/ceiling anchors + a perturbed-cache accuracy)

In [ ]:
@torch.no_grad()
def anchors(R,items):
    fl=ce=0; n=0
    for it in items:
        g=it["ans"]
        v,l,_=R.ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"]); fl+=hit(R.greedy(v,l),g)
        v,l,_=R.ctx_cache(make_prompt(it["full"],it["q"]),  CFG["max_ctx"]); ce+=hit(R.greedy(v,l),g)
        n+=1
    return fl/max(1,n), ce/max(1,n)

@torch.no_grad()
def transfer_of(R,items,make_prefix):
    '''make_prefix(it)->(prefix_vec,last_id): the perturbed full-context cache to read.'''
    a=0;n=0
    for it in items:
        pv,last=make_prefix(it); a+=hit(R.greedy(pv,last),it["ans"]); n+=1
    return a/max(1,n)

@torch.no_grad()
def mean_key_cos(R,va,vb):
    cs=[]
    for l in range(R.L):
        a=va[l,0,:,:R.kdim].reshape(-1); b=vb[l,0,:,:R.kdim].reshape(-1)
        cs.append(F.cosine_similarity(a,b,0).item())
    return float(np.mean(cs))

## 6. LEG A + B — depth-budget sweep on Qwen2.5-1.5B

In [ ]:
R=Reader(BASE)
Kbank,Vbank,Kn=build_bank(R)
fl,ce=anchors(R,eval_qa)
print(f"[{BASE}] floor={fl:.3f} ceiling={ce:.3f}",flush=True)
RES["legs"]["anchors_base"]=dict(floor=fl,ceiling=ce)

# precompute per-item own + perturbed full-context caches once
def make_own(it):  return R.ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
ELL=sorted(set([0,1,2,3,4,6,8,12,16,20,24,R.L]))
ELL=[l for l in ELL if l<=R.L]
legA={"ell":ELL,"nn1":[],"gauss":[],"cos_nn1":None,"cos_gauss":None}
# fix a matched gaussian cos to nn1's mean cos so A's two curves are fidelity-matched
# first measure nn1 mean cos on a few items
cos_nn1=[]
for it in eval_qa[:20]:
    own,last,_=make_own(it); pv=nn1(R,own,Kbank,Vbank,Kn); cos_nn1.append(mean_key_cos(R,own,pv))
cos_nn1=float(np.mean(cos_nn1)); legA["cos_nn1"]=cos_nn1
print(f"nn1 mean key-cos = {cos_nn1:.3f} (gauss matched to this)",flush=True)

for l in ELL:
    def mk_nn1(it,ll=l):
        own,last,_=make_own(it); pert=nn1(R,own,Kbank,Vbank,Kn)
        return depth_budget_vec(R,own,pert,ll),last
    def mk_g(it,ll=l):
        own,last,_=make_own(it); pert=gauss_to_cos(R,own,cos_nn1)
        return depth_budget_vec(R,own,pert,ll),last
    a_nn1=transfer_of(R,eval_qa,mk_nn1)
    a_g  =transfer_of(R,eval_qa,mk_g)
    legA["nn1"].append(a_nn1); legA["gauss"].append(a_g)
    print(f"  l={l:2d}: nn1={a_nn1:.3f}  gauss(cos{cos_nn1:.3f})={a_g:.3f}",flush=True)
    RES["legs"]["A_depth_budget"]=legA
    json.dump(RES,open(RESPATH,"w"),indent=1)

legB=dict(l1_nn1=legA["nn1"][ELL.index(1)], l1_gauss=legA["gauss"][ELL.index(1)],
          full_nn1=legA["nn1"][ELL.index(R.L)], floor=fl, ceiling=ce)
RES["legs"]["B_single_layer"]=legB
print(f"[LEG B] l=1 nn1={legB['l1_nn1']:.3f} vs full nn1={legB['full_nn1']:.3f} "
      f"(floor {fl:.3f}, ceiling {ce:.3f})",flush=True)
json.dump(RES,open(RESPATH,"w"),indent=1)

## 7. LEG C — anisotropy: aligned vs random perturbation at equal cosine

In [ ]:
dirs=sensitivity_dirs(R,eval_qa,CFG["aniso_rank"])
legC={"cos_budgets":CFG["cos_budgets"],"aligned":[],"random":[],"ok":dirs is not None}
if dirs is None:
    print("[LEG C] could not access k_proj/q_proj for this model; skipping",flush=True)
else:
    for cb in CFG["cos_budgets"]:
        def mk_al(it,c=cb):
            own,last,_=make_own(it); return aniso_to_cos(R,own,c,dirs),last
        def mk_rn(it,c=cb):
            own,last,_=make_own(it); return gauss_to_cos(R,own,c),last
        a_al=transfer_of(R,eval_qa,mk_al); a_rn=transfer_of(R,eval_qa,mk_rn)
        legC["aligned"].append(a_al); legC["random"].append(a_rn)
        print(f"  cos={cb:.3f}: aligned={a_al:.3f}  random={a_rn:.3f}  gap={a_rn-a_al:+.3f}",flush=True)
        RES["legs"]["C_anisotropy"]=legC
        json.dump(RES,open(RESPATH,"w"),indent=1)

## 8. LEG D — genuine tolerance radius vs L across real models

In [ ]:
legD={"models":[]}
for name in LADDER:
    try:
        Rd=Reader(name)
    except Exception as e:
        print(f"[{name}] load failed: {e}",flush=True); continue
    items=eval_qa[:CFG["ladder_neval"]]
    Kb,Vb,Knn=build_bank(Rd) if True else (None,None,None)
    fld,ced=anchors(Rd,items)
    def mk_nn1(it):
        own,last,_=Rd.ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
        return nn1(Rd,own,Kb,Vb,Knn),last
    def mk_id(it):
        own,last,_=Rd.ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
        return own,last
    a_nn1=transfer_of(Rd,items,mk_nn1)
    a_id =transfer_of(Rd,items,mk_id)  # identity == ceiling sanity
    surv=(a_nn1-fld)/max(1e-6,(ced-fld))   # fraction of the floor->ceiling band retained
    row=dict(name=name,L=Rd.L,n_kv=Rd.n_kv,group=Rd.group,floor=fld,ceiling=ced,
             nn1=a_nn1,identity=a_id,survival=surv)
    legD["models"].append(row)
    print(f"[{name}] L={Rd.L} floor={fld:.3f} ceil={ced:.3f} nn1={a_nn1:.3f} "
          f"id={a_id:.3f} survival={surv:.3f}",flush=True)
    RES["legs"]["D_radius_vs_L"]=legD
    json.dump(RES,open(RESPATH,"w"),indent=1)
    del Rd; torch.cuda.empty_cache()

## 9. Verdict + plots

In [ ]:
print("\n================ DEPTH-SWEEP VERDICT ================",flush=True)
b=RES["legs"].get("B_single_layer",{})
if b:
    print(f"LEG B (per-layer sharpness): after ONE corrupted layer, nn1 transfer="
          f"{b['l1_nn1']:.3f} vs floor {b['floor']:.3f} / ceiling {b['ceiling']:.3f}.",flush=True)
    frac=(b['l1_nn1']-b['floor'])/max(1e-6,b['ceiling']-b['floor'])
    print(f"   -> one layer already destroys {100*(1-frac):.0f}% of the band => "
          f"{'STRONGER-THAN-DEPTH: per-layer sharpness dominates' if frac<0.5 else 'depth-limited'}",flush=True)
c=RES["legs"].get("C_anisotropy",{})
if c and c.get("ok"):
    import numpy as _np
    gaps=[r-a for a,r in zip(c["aligned"],c["random"])]
    print(f"LEG C (anisotropy, depth-FREE): mean aligned-vs-random gap at equal cos = "
          f"{_np.mean(gaps):+.3f} (max {max(gaps):+.3f}). Equal fidelity, different transfer "
          f"=> cosine blind to the leverage axis (explains O2).",flush=True)
d=RES["legs"].get("D_radius_vs_L",{})
if d:
    print("LEG D (radius vs L):",flush=True)
    for r in sorted(d["models"],key=lambda x:x["L"]):
        print(f"   L={r['L']:2d} {r['name']:24s} survival={r['survival']:.3f}",flush=True)
    print("   depth leg real if survival falls with L; residual at small L = depth-free wall.",flush=True)
json.dump(RES,open(RESPATH,"w"),indent=1); print("saved",RESPATH,flush=True)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# A/B depth-budget
A=RES["legs"]["A_depth_budget"]; fl=RES["legs"]["anchors_base"]["floor"]; ce=RES["legs"]["anchors_base"]["ceiling"]
ax[0].plot(A["ell"],A["nn1"],"-o",color="#d95f0e",label="nn1 (rank-1 neighbour)")
ax[0].plot(A["ell"],A["gauss"],"-s",color="#2c7fb8",label=f"gauss @cos{A['cos_nn1']:.3f}")
ax[0].axhline(fl,ls="--",color="#999"); ax[0].axhline(ce,ls="--",color="#000")
ax[0].set_xlabel("# corrupted layers l (rest genuine)"); ax[0].set_title("LEG A/B depth-budget: damage vs depth")
ax[0].legend(fontsize=8); ax[0].grid(ls=":",alpha=.4)
# C anisotropy
C=RES["legs"].get("C_anisotropy",{})
if C.get("ok"):
    ax[1].plot(C["cos_budgets"],C["aligned"],"-o",color="#d95f0e",label="aligned (sensitive dirs)")
    ax[1].plot(C["cos_budgets"],C["random"],"-s",color="#2c7fb8",label="random (same cos)")
    ax[1].axhline(fl,ls="--",color="#999"); ax[1].axhline(ce,ls="--",color="#000")
    ax[1].invert_xaxis(); ax[1].set_xlabel("cosine to C* (fidelity)"); ax[1].set_title("LEG C anisotropy (depth-free)")
    ax[1].legend(fontsize=8); ax[1].grid(ls=":",alpha=.4)
# D radius vs L
D=RES["legs"].get("D_radius_vs_L",{})
if D:
    rows=sorted(D["models"],key=lambda x:x["L"])
    ax[2].plot([r["L"] for r in rows],[r["survival"] for r in rows],"-o",color="#41ab5d")
    for r in rows: ax[2].annotate(r["name"].split("/")[-1],(r["L"],r["survival"]),fontsize=7)
    ax[2].set_xlabel("depth L"); ax[2].set_ylabel("band survival (nn1)"); ax[2].set_title("LEG D radius vs depth")
    ax[2].grid(ls=":",alpha=.4)
fig.tight_layout(); fig.savefig("figures/depth_curves.png",dpi=150); plt.show()